In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from torch.utils.data import TensorDataset
import torch.nn.functional as F
from torch.optim import AdamW

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32).flatten()
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)



In [ ]:
# 2. Create TensorDataset objects
train_dataset=TensorDataset(X_train,y_train)
test_dataset=TensorDataset(X_test,y_test)



In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(
train_dataset,
batch_size=32,
shuffle=True,
num_workers=2)
test_loader = DataLoader(
test_dataset,
batch_size=32,
shuffle=False,)


In [ ]:
# 4. Print shape of one batch



In [ ]:
# 5. Display sample images
images,labels=next(iter(train_dataset))
plt.figure(figsize=(20,4))
for i in range(2):
  plt.subplot(2, 3, i + 1)
  plt.imshow(images[i].squeeze(), cmap='gray')
  plt.axis('off')
  plt.tight_layout()
  plt.show()



In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(NN4Layer, self).__init__()

    self.layer1 = nn.Linear(input_dim, hidden_dim)

    self.layer2 = nn.Linear(hidden_dim, hidden_dim)

    self.layer3 = nn.Linear(hidden_dim, hidden_dim)

    self.layer4=nn.Linear(hidden_dim,output_dim)

    self.relu = nn.ReLU()

  def forward(self, x):

    z1 = self.layer1(x)
    a1 = self.relu(z1)

    z2 = self.layer2(a1)
    a2 = self.relu(z2)
    z3=self.layer3(a2)
    a3=self.relu(z3)
    output = self.layer4(a3)
    return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  model.train()
  running_loss = 0.0
  for X_batch, y_batch in train_loader:

    X_batch = X_batch.view(X_batch.size(0), -1).to(device)
    y_batch = y_batch.to(device)

    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    running_loss += loss.item()


In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  model.eval()
  running_loss = 0.0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:

      X_batch = X_batch.view(X_batch.size(0), -1).to(device)
      y_batch = y_batch.to(device)

      outputs = model(X_batch.viewsview(images.size(0), -1))
      loss = criterion(outputs, y_batch)
      running_loss += loss.item()




In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim=2000
hidden_dim=14
output_dim=101
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)
loss=nn.MSELoss()
num_epochs=20
learning_rate=0.01
optimizer=AdamW(model.parameters(),learning_rate)


In [ ]:
# Task 5: Start training for 20 epochs:
train_losses=[]
test_losses=[]
print('Starting Training...')
for epoch in range(num_epochs):

  train_loss = train_one_epoch(model, optimizer, loss, train_loader, device)

  val_loss, val_accuracy = validate(model, loss, test_loader, device)
  train_losses.append(train_loss)
  test_losses.append(val_loss)
  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, 'b-o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: